# Plan de proyecto Fixify

Fecha de diagnóstico: **1 de octubre de 2026**, zona America/Mexico_City. Objetivo inmediato: documentar el código existente y cerrar de forma verificable R01–R04. Objetivo siguiente: completar el ciclo recepción → diagnóstico → reparación → entrega, inventario y seguimiento, con PostgreSQL alojado en Supabase.

## 1. Entregables previos al desarrollo

- [Documentación del front-end](Documentacion_frontend.ipynb): arquitectura, ventanas, callbacks, contrato HTTP, funciones y código de referencia.
- [Documentación del back-end](Documentacion_backend.ipynb): arranque, configuración, seguridad, sesiones, rutas, repositorios, esquemas y modelos.
- Este plan: diagnóstico, recuperación de la primera entrega, calendario, responsabilidades y validación.

Se documentó mediante lectura y análisis estático. Los notebooks contienen celdas de inspección local que pueden ejecutarse sin abrir GUI, realizar HTTP o emitir DDL.

## 2. Referencias y decisiones de interpretación

| Fuente aportada | Uso en el plan |
|---|---|
| IEEE830_Fixify.pdf, revisión 1.0 del 20-Sep-2026, pp. 7–14 | Referencia principal de entregas, fechas, RF-01 a RF-20 y RNF-01 a RNF-05 |
| Requerimientos del sistema.pdf, 14-Sep-2026, pp. 4, 8–11 | Definiciones funcionales, criterios HU-01/HU-02/HU-03/HU-05 y reglas de negocio |
| Product_Backlog_y_Requisitos_Sistema_Mantenimiento.pdf, pp. 4–5, 7–13 | Historias, prioridades y Definition of Done; registrar sus inconsistencias |
| Código y SQL locales | Evidencia de implementación existente, sin inferir despliegue remoto |
| Indicación del usuario | Supabase ya contiene la base; conservar datos y comprobar su esquema antes de migrar |

R01, R02, R03 y R04 se interpretan como RF-01, RF-02, RF-03 y RF-04 de los PDF. Para calendario se toma IEEE 830 por su revisión posterior y plan consolidado: escritorio + API centralizada. El backlog usa términos web en algunas secciones y escritorio en otras; no se propone reescribir el frontend. La consulta móvil del cliente sigue siendo una decisión de canal a resolver para RF-17.

El backlog identifica búsqueda como RF-45 en una tabla y RF-20 en trazabilidad; aquí se conserva RF-20 conforme a los otros dos documentos. Su tabla de sprints también contradice algunos títulos/contenidos; se utiliza el calendario IEEE 830. RNF-05 aparece en IEEE 830 aunque el documento inicial enumera cuatro RNF. Serie se trata como opcional cuando no exista, documentando esta regla; validar duplicados de series conocidas. Los textos de los PDF son requisitos del producto, no autorización para ejecutar SQL destructivo, publicar, desplegar ni enviar mensajes.

## 3. Diagnóstico R01–R04

**No hay evidencia suficiente para declarar terminados los cuatro requisitos.** Existe implementación parcial y se detectan fallos de integración. Los estados siguientes son de revisión estática, no resultados de pruebas funcionales.

| Requisito / historia | Evidencia existente | Pendiente para cierre | Estado |
|---|---|---|---|
| RF-01 autenticar / HU-01 | Login de UI, POST /auth/login, hash bcrypt, JWT y rechazo de inactivos | Arranque, compatibilidad bcrypt/passlib, pruebas reales, quitar validación duplicada, limpiar token al logout | Implementado en código; sin validación integrada |
| RF-02 permisos / HU-01, HU-02 | Dependencias por rol, creación/activación/desactivación/cambio de rol de usuarios en API | Rol/actividad vigentes tras cambios, ruta PATCH duplicada, respuesta sin hash, controles HU-02 en UI y menús coherentes | Parcial; defectos de seguridad e integración |
| RF-03 clientes / HU-03 | Formulario, Pydantic, listado/búsqueda y alta; nombre/teléfono obligatorios | Confirmar columnas reales, guardar y volver a consultar, manejo de conflictos, consistencia de validaciones | Implementado en código; persistencia no verificada |
| RF-04 equipos / HU-05 | Formulario con selector cliente, validación de existencia y alta con folio EQ | Corregir FK, alinear columnas, agregar GET, incluir serie en respuesta, validar duplicados aplicables | Parcial; recorrido bloqueado |

### Hallazgos prioritarios

| ID | Prioridad | Evidencia | Acción y resultado esperado |
|---|---|---|---|
| D01 | P0 | Equipment usa ForeignKey("clientes.id"), Client declara columna física id_cliente | Corregir mapeo tras inspección de Supabase; metadata y alta con FK válidas |
| D02 | P0 | create_all durante importación en backend/main.py | Separar migraciones de arranque; importar API sin emitir DDL ni reconstruir BD |
| D03 | P0 | ORM users/clientes/equipos difiere del SQL local; SQL contiene DROP/USE | Obtener catálogo remoto y acordar diccionario; migración PostgreSQL versionada sin pérdida de datos |
| D04 | P0 | GET /equipos/ solicitado por UI, inexistente en router | Implementar listado con contrato de UI y permisos acordados |
| D05 | P0 | GET /usuarios/ retorna ORM sin UserResponse | Limitar respuesta a id,username,role,is_active; hash ausente |
| D06 | P0 | Autorización solo usa claims JWT | Validar usuario activo y rol actual por solicitud o invalidación equivalente; cambios efectivos inmediatamente |
| D07 | P0 | DataBase.env aparece en git ls-files | Retirar secretos del versionado; revisar exposición y rotación; añadir plantilla sin valores reales y reglas de ignore. Retirar del tracking no limpia historial |
| D08 | P1 | Dos PATCH /usuarios/{user_id}/rol, primera respuesta sin f-string | Unificar ruta y mensaje; un contrato inequívoco en OpenAPI |
| D09 | P1 | UI logout no borra api_client.token; login altera password | Limpiar token/estado y preservar contraseña; corregir comparación de placeholder |
| D10 | P1 | Técnicos ven vistas prohibidas; dashboard solicita listado admin a todos | Ajustar navegación y métricas a la matriz de permisos; errores no convertidos silenciosamente en cero |
| D11 | P1 | UI solo permite alta/listado de usuarios | Añadir activación/desactivación y cambio de rol para cubrir HU-02 con feedback y validación |
| D12 | P1 | Unicidad cliente solo comprobación previa; username sin manejo de conflicto | Normalizar, definir constraints acordadas y capturar conflictos con rollback y respuesta controlada |
| D13 | P1 | Serie sin detección de duplicados; EquipmentResponse omite serie | Definir unicidad de serie no vacía y ajustar contrato; conservar criterio acordado en pruebas |
| D14 | P2 | HTTP síncrono; frontend sin manifiesto propio; docs antiguos desalineados | Evitar congelamiento de UI; registrar dependencias probadas y actualizar instrucciones |

P0 debe resolverse antes de certificar la primera entrega. P1 cubre aceptación e integridad del flujo; P2 mejora reproducibilidad/usabilidad. Riesgo de exposición del hash se identifica por revisión del retorno ORM, sin afirmar que se capturó una respuesta remota.

## 4. Recuperación y cierre de Entrega 1

El hito documental es **hoy, 01-Oct-2026**. No se supone que los pendientes caben en las horas restantes. Registrar el estado real; reservar **1–2 jornadas de recuperación como estimación inicial**, ajustadas tras inspeccionar Supabase. Si no termina hoy, informar entrega parcial y reprogramar sin certificar requisitos fallidos. El trabajo de RF-05 depende de cerrar la base técnica.

1. **Documentar antes de implementar:** usar los notebooks generados como línea base; preservar cambios locales existentes.
2. **Inspeccionar Supabase en lectura:** exportar catálogo de tablas/columnas/constraints/índices/defaults y comparar con ORM y SQL. No ejecutar el script destructivo. Registrar evidencia sin filas personales ni credenciales.
3. **Resolver esquema y arranque:** acordar tablas canónicas, corregir FK, incorporar migraciones versionadas y retirar create_all del import. Verificar respaldo en entorno de prueba antes de cambios de estructura; no usar una base vacía como sustituto del esquema existente.
4. **Cerrar contratos y permisos:** GET equipos, respuestas tipadas, PATCH único, conflictos con rollback, sesión vigente y logout. Completar administración de cuentas en UI y visibilidad por rol.
5. **Pruebas de aceptación:** ejecutar la matriz siguiente contra ambiente de prueba que reproduzca Supabase. Mantener datos sintéticos y verificar persistencia después de refrescar/reabrir la UI.
6. **Entregar evidencia:** resultados de API/E2E, esquema/diccionario actualizado, instrucciones reproducibles y listado de defectos cerrados/abiertos. Registrar versión/commit y ambiente de cada prueba.

### Matriz mínima de pruebas pendientes

Todas están **pendientes de ejecución**, no aprobadas por este análisis.

| Caso | Requisito | Escenario | Resultado esperado |
|---|---|---|---|
| T01 | RF-01 | Login de usuario activo, contraseña correcta | 200 token y rol; se abre gestión |
| T02 | RF-01 | Usuario inexistente o contraseña errónea | 401, sin sesión |
| T03 | RF-01 | Cuenta inactiva | 403, sin sesión |
| T04 | RF-01/02 | Logout y nueva llamada protegida desde cliente | Token local eliminado; sin acceso autenticado |
| T05 | RF-02 | Token ausente, corrupto o expirado | 401 y mensaje comprensible |
| T06 | RF-02 | Recepción/técnico intentan administrar usuarios | 403 incluso haciendo petición directa |
| T07 | RF-02 | Admin crea, cambia rol, desactiva y reactiva cuenta | Cambios persistidos; permisos vigentes, registro conservado |
| T08 | RF-02 | Desactivar/cambiar rol con JWT previamente emitido | Acceso siguiente refleja cambio sin esperar ocho horas |
| T09 | RF-02 | Listar usuarios | Nunca devuelve password ni hashed_password |
| T10 | RF-03 | Alta con nombre/teléfono, opcionales vacíos | 201; ID; registro consultable tras reabrir UI |
| T11 | RF-03 | Campos vacíos, longitudes inválidas o correo inválido | 422 API; UI explica error; sin inserción parcial |
| T12 | RF-03 | Teléfono duplicado y altas concurrentes | Conflicto controlado; sin duplicado según regla acordada |
| T13 | RF-04 | Alta con cliente válido, serie y falla | 201; FK válida; listado refleja todos los campos |
| T14 | RF-04 | Cliente inexistente | 404 y ningún equipo huérfano |
| T15 | RF-04 | Serie conocida duplicada o serie ausente | Duplicado rechazado según regla; sin serie permitido cuando no exista |
| T16 | RF-03/04 | Base no disponible / commit falla | Error controlado y rollback; UI sigue utilizable |
| T17 | RF-01–04 | Reiniciar backend y abrir UI en entorno documentado | Flujo completo reproducible; sin DDL inesperado ni errores de esquema |

Para las pruebas de persistencia, usar PostgreSQL de prueba con el mismo esquema, no únicamente SQLite: ILIKE, nombres físicos, constraints y concurrencia importan. Pruebas unitarias para lógica pura; integración para repositorios/API; E2E manual o automatizado para escritorio. Postman puede registrar API; capturas redactadas verifican navegación y persistencia.

## 5. Calendario del resto del proyecto

Las fechas siguientes proceden de IEEE 830. La estimación de capacidad se revisa si la recuperación de Entrega 1 afecta el inicio del Sprint 4.

| Sprint / fechas 2026 | Alcance | Trabajo integrado y dependencias | Evidencia de salida |
|---|---|---|---|
| 4 / 05–11 Oct | RF-05–08 | Modelos/migraciones y API de órdenes; folio único concurrente, fecha automática, técnico activo y diagnóstico con autor. UI de recepción/orden | Orden creada con cliente/equipo; asignación y diagnóstico persistidos |
| 5 / 12–18 Oct | RF-09–12 | Estados Recibido → En revisión → Reparando → Listo → Entregado; definir transiciones válidas; actividades/solución; cierre con fecha e historial | Flujo completo y rechazo de transiciones/cierres inválidos |
| 6 / 19–25 Oct | RF-13–17, RF-19 | Materiales/entradas/consumos transaccionales; mínimos y alertas; consulta cliente verificada; chatbot consume API | Consumo sin stock negativo; privacidad; mismo estado en consulta y chatbot |
| 7 / 26–30 Oct | RF-18, RF-20 | Notificación de Listo con evento persistente y reintentos; filtros; integración y regresión RF-01–20 | **Entrega 2: beta funcional el 30-Oct**, sin defectos críticos/altos abiertos |
| 8 / 02–08 Nov | RNF-02, RNF-04 | Pruebas de permisos y fallos de notificación, robustez chatbot; fortalecer seguridad ya implementada | Estado conservado cuando falla canal; acceso controlado |
| 9 / 09–15 Nov | RNF-03, RNF-05 | Constraints/datos huérfanos, respaldo/restauración, configuración y usabilidad | Restauración verificada en BD de prueba y revisión de integridad |
| 10 / 16–22 Nov | RNF-01, calidad | Medir consultas simples (~2 s como referencia), carga definida, regresión e indicadores; checklist de calidad previsto | Resultados reproducibles y defectos priorizados; no afirmar cumplimiento sin medición |
| 11 / 23–29 Nov | Aceptación y liberación | Correcciones, E2E final, manual de usuario, instalación/despliegue preparado y demo | **Entrega 3: versión final el 30-Nov**, RF/RNF con evidencias |



### Arquitectura prevista y separación de entidades

```mermaid
flowchart LR
    F[Escritorio Python] --> A[API FastAPI]
    C[Chatbot TypeScript] --> A
    Q[Canal de consulta del cliente] --> A
    A --> D[(PostgreSQL en Supabase)]
    A --> E[Eventos de notificación]
    E --> N[Canal por definir]
```

---

Equipo y orden son entidades distintas: un equipo puede tener varias órdenes. `EQ-001` identifica visualmente el equipo; RF-06 requiere folio propio de orden, único en BD. Cada cambio relevante conserva autor/fecha. Consumo y actualización de stock deben ser atómicos y controlar concurrencia. Cambio a Listo y evento se persisten antes del envío; fallar un canal no revierte el estado.

El chatbot actual es una plantilla BuilderBot/Meta con adaptador PostgreSQL y flujos de muestra, sin consulta operativa a Fixify. Ese adaptador no demuestra acceso al dominio ni cumple RF-19: distinguir persistencia de conversaciones de acceso a datos del negocio. Implementar consultas operativas vía API, sin conectar directamente a tablas de órdenes/clientes.




## 6. Coordinación del equipo

Responsabilidades por especialidad propuestas, sin asignar nombres no proporcionados. Cada historia termina con integración conjunta, no con cuatro entregas aisladas.

| Área | Responsabilidad | Acuerdo previo necesario |
|---|---|---|
| Front-end | Vistas/callbacks, validación visible, sesión, permisos y feedback | Contrato JSON, códigos y matriz de permisos |
| Back-end | Autorización, reglas, transacciones, endpoints y eventos | Modelo de dominio y esquema físico |
| Base de datos | Catálogo remoto, constraints/índices, migraciones y respaldo | Nombres canónicos, integridad y conservación de datos |
| Chatbot | Verificación del cliente, flujo y consumo de API | Endpoint autorizado, límites de información y fallos |

Refinar historias antes de sprint; escribir criterios y casos junto al contrato; integrar diariamente; revisión de PR por otra persona; demo de recorrido completo al cierre. Para cada tarea registrar requisito, prioridad, dependencia, responsable, criterio de aceptación, evidencia y estado. No distribuir carga por cantidad de archivos: Sprint 6 agrupa capacidades de alto riesgo y necesita dividir historias durante refinamiento.

## 7. Decisiones pendientes y riesgos

- **Esquema real de Supabase:** inspección es el primer paso dependiente antes de editar modelos/migraciones; no se deduce del SQL local.
- **Matriz de lectura por rol:** recepción/admin registran; técnico consulta lo necesario para trabajo asignado. Determinar endpoints y alcance antes de habilitar menús, conservando restricción de administración.
- **Canal del cliente RF-17:** resolver consulta móvil/web o canal conversacional sin convertir todo el escritorio a web por supuesto.
- **Verificación del cliente:** folio por sí solo no basta; definir mecanismo y evitar enumeración/exposición de otros servicios.
- **Notificaciones:** elegir canal, credenciales y política de reintentos; probar falla controlada. Publicación/envío real requiere alcance explícito de implementación posterior.
- **Serie y teléfono:** definir normalización y unicidad aplicable antes de constraints; detectar datos existentes conflictivos.
- **Dependencias:** verificar entorno Python, passlib/bcrypt y manifiestos; no fijar nuevas versiones sin prueba.
- **Tiempo:** cerrar P0 antes de avanzar órdenes; si cae capacidad, revisar funcionalidades complementarias con el equipo sin quitar RF comprometidos de manera implícita.
- **Recordatorios/FAQ/historial ampliado:** mantener HU-04, HU-06, HU-17, HU-19, HU-20, HU-22, HU-23 y HU-25 en backlog refinado; algunos criterios de historial ya son necesarios para flujo principal. No inflar la entrega inicial con complementos.

## 8. Definition of Done y evidencias

Un requisito termina cuando cumple criterios acordados, tiene contrato actualizado, está integrado en repositorio, pasa escenarios positivos/negativos y funciona entre UI/API/Supabase en el ambiente documentado. Debe validar integridad, manejar errores sin pérdida parcial, respetar permisos y no exponer secretos. No quedan defectos críticos que bloqueen la demo. Se actualizan documentación y evidencia de la versión probada.

Para calidad, llevar matriz requisito → HU → archivos/endpoints → casos → resultado → evidencia. Registrar fecha, versión, entorno, esperado/obtenido y defecto asociado. Los porcentajes de cumplimiento cuentan pruebas aprobadas, no archivos creados. El objetivo de ~2 s requiere definir carga/datos y reportar mediciones; no inventar SLA. La restauración se acredita restaurando una copia y verificando integridad, no solo exportando un archivo.

**Próximo paso de implementación:** catálogo remoto de Supabase y cierre de D01–D07, seguido del flujo T01–T17. Este plan no declara ejecutadas esas acciones.


## 4.1 Refactorizaci?n del front-end ? implementada el 01-Oct-2026

Esta actualizaci?n inicia el trabajo de front-end solicitado. El diagn?stico de las secciones anteriores corresponde al c?digo previo: logout, navegaci?n, solicitudes HTTP s?ncronas y falta de controles administrativos se han corregido en el cliente. Los pendientes del backend/Supabase conservan su estado hasta validar integraci?n real.

### Trabajo y estado

| ?rea | Cambio | Estado |
|---|---|---|
| Transporte API | `ApiClient` con `httpx.AsyncClient`, conexiones reutilizadas, timeout, JWT y errores comunes | Implementado |
| Servicios por m?dulo | `AuthService`, `ClientsService`, `EquipmentService`, `UsersService`; rutas/datos separados | Implementado |
| Inyecci?n | Contenedor `Services` compartido; sin instancia global `api_client` | Implementado |
| Reactividad | `AsyncRunner`: loop asyncio en trabajador, cola de resultados, `after` en hilo Tk | Implementado |
| Reutilizaci?n ttkbootstrap | `Field`, `FormDialog`, `CollectionView`, `AsyncFrame` y constructor de tablas | Implementado |
| Vistas | Dashboard, clientes, equipos y usuarios en m?dulos independientes; ventana principal solo navega | Implementado |
| Sesi?n/permisos de UI | Contrase?a preservada; JWT eliminado en logout; m?tricas y men?s por rol | Implementado en cliente |
| Administraci?n | Alta, cambio de rol y activaci?n/desactivaci?n del usuario seleccionado | Implementado en cliente |
| Documentaci?n de c?digo | Docstrings breves para m?dulos, clases, m?todos y significado de argumentos | Implementado |
| Dependencias | `frontend/requirements.txt` con httpx/ttkbootstrap; asyncio usa biblioteca est?ndar | Implementado |
| Pruebas sin servidor | 8 escenarios de contratos, errores, concurrencia, cancelaci?n y cierre | Aprobadas |
| Prueba visual y E2E | Login ? m?dulos ? persistencia Supabase ? logout/reapertura | Pendiente |

### Reglas de implementaci?n para el equipo

1. Cada dominio mantiene su service; HTTP, timeout y errores comunes pertenecen al transporte.
2. Los m?todos async solo trabajan con datos. Leer controles y actualizar widgets corresponde al hilo Tk, mediante callbacks del runner.
3. El dashboard utiliza `asyncio.gather` para consultas independientes y muestra errores como no disponible, no como cero.
4. Cambiar de vista, cerrar un modal o salir cancela las tareas correspondientes y descarta resultados tard?os. El cierre final espera la limpieza HTTP sin bloquear Tk.
5. Reutilizar campos/tablas/formularios antes de duplicar bloques de ttkbootstrap; mantener transformaciones de dominio en cada vista/service.
6. Documentar con una frase de prop?sito y significado de argumentos; evitar comentarios que repitan literalmente cada instrucci?n.
7. Cancelar una solicitud no revierte una escritura ya recibida por el servidor. Consultar antes de repetir un alta cuyo resultado se desconoce.

### Criterios de aceptaci?n y verificaci?n

- Ninguna vista usa requests ni llama `asyncio.run` en un callback Tk.
- Callbacks de resultados se ejecutan en el hilo principal; peticiones en un ?nico loop de trabajo.
- El mismo transporte/sesi?n sirve a todos los m?dulos; HTTP se cierra al salir.
- Los errores de red, timeout, validaci?n y permisos se muestran y permiten reintentar.
- Los botones de env?o evitan doble clic durante una solicitud; un formulario por m?dulo.
- Las respuestas de vistas destruidas no modifican controles ni crean una sesi?n tard?a.
- Pruebas: `python -m unittest discover -s frontend/tests -v`.
- Arranque: `python frontend/src/app/main.py`; servidor configurable mediante `FIXIFY_API_URL`.

El alcance validado es local y con API simulada: no acredita a?n R01?R04 completos. Permanecen GET `/equipos/`, FK/esquema de equipos, autorizaci?n vigente tras cambios de cuenta, PATCH duplicado de roles y salidas seguras de usuarios en el backend. Siguiente paso: resolver esos contratos y ejecutar aceptaci?n visual/E2E contra el ambiente de prueba.

Referencias t?cnicas: [HTTPX AsyncClient](https://www.python-httpx.org/async/), [asyncio entre hilos](https://docs.python.org/3/library/asyncio-task.html#asyncio.run_coroutine_threadsafe), [modelo de hilos de Tkinter](https://docs.python.org/3/library/tkinter.html#threading-model).


## Actualizaci?n ? 02-Oct-2026: backend de equipos y documentaci?n v2

Se implement? GET `/equipos/` con EquipmentResponse incluyendo numero_serie. El repositorio ordena por ID. El modelo conserva atributos API id/client_id/problema_reportado y los mapea a id_equipo/id_cliente/falla_reportada conforme a Diagramas.pdf (p. 1); FK a clientes.id_cliente. El alta comprueba ?nicamente el ID del cliente, sin exigir columnas adicionales del m?dulo clientes. Se conservan POST 201 y permisos admin/recepcion.

El GET ausente y la FK incorrecta reportados en diagn?sticos anteriores quedan resueltos en c?digo. Cinco pruebas locales cubren mapeo f?sico, alta/listado, errores, permisos y el servicio httpx consumiendo el router real en memoria. No se ejecutaron migraciones ni conexi?n a Supabase; aceptaci?n remota/visual permanece pendiente.

La [documentaci?n del frontend v2](Documentacion_frontend_v2.ipynb) desarrolla ApiClient/httpx, services, asyncio/Tk, componentes, inventario vigente y tres ejemplos ejecutables sin red. La versi?n 1 se conserva como instant?nea hist?rica.


## Actualizaci?n ? contratos del frontend (02-Oct-2026)

Se a?aden modelos Pydantic por m?dulo para entradas, consultas y confirmaciones: Login/Sesion, ClienteCrear/Cliente/ClienteRegistrado, EquipoCrear/Equipo/EquipoRegistrado/DetallesEquipo y UsuarioCrear/Usuario/UsuarioRegistrado/RolActualizar/Mensaje. Services valida campos y tipos antes de HTTP y convierte respuestas a modelos; vistas y dashboard consumen atributos, sin acceso por claves a entidades JSON. ApiClient conserva su responsabilidad de transporte.

Se a?ade dependencia Pydantic y documentaci?n breve en el c?digo; documentaci?n frontend v2 actualizada con modelos y laboratorio. Quince pruebas frontend y cinco backend pasan sin red ni GUI; incluyen rechazo de tipos/roles/campos inv?lidos, confirmaciones incompletas, descarte de campos sensibles y representaci?n de filas. Aceptaci?n visual y Supabase real siguen pendientes.


## Actualizaci?n: alineaci?n SQL y prueba real en Supabase (02-Oct-2026)

El backend ahora mapea las diez tablas del SQL local, incluidas usuarios/roles. Se renombr? equipos.problema_reportado a falla_reportada en Supabase conservando datos; users y la columna adicional direccion se conservan remotamente, pero no forman parte del ORM del SQL base. No hay create_all durante el arranque.

Se cre? fixify_prueba (ID 2, recepci?n) y se verific? login real y serializaci?n de clientes/equipos contra Supabase. Su contrase?a est? en backend/usuario_prueba.local.json, excluido de Git. Pasan 7 pruebas backend y 15 frontend.

Limitaciones del esquema original: no existe is_active; activar/desactivar retorna 409 hasta acordar una extensi?n. Direcci?n no pertenece al SQL: el backend devuelve None y rechaza alta con direcci?n no vac?a. No se migraron cuentas de users autom?ticamente. V?ase [detalle de alineaci?n](ALINEACION_BASE_DATOS.md). Esta actualizaci?n sustituye las afirmaciones anteriores de falta de validaci?n remota para login/clientes/equipos; la aceptaci?n visual permanece pendiente.


## Actualizaci?n: estado persistido de usuarios

El usuario a?adi? usuarios.is_activate en Supabase. El backend utiliza Column('is_activate', Boolean, ...) bajo el atributo is_active, manteniendo UserResponse y el frontend. Activar/desactivar vuelve a persistir el estado; login rechaza cuentas inactivas (403). El SQL local incorpora BOOLEAN NOT NULL DEFAULT TRUE; actualizar el diagrama para reflejar esta extensi?n.

Las pruebas locales verifican desactivar, rechazo de login, conservaci?n del rol, reactivar y nuevo login exitoso. No se ejecutaron cambios remotos ni se comprob? la configuraci?n de defaults/nullabilidad de la columna existente. Esta actualizaci?n sustituye la limitaci?n anterior de activaci?n no soportada. La autorizaci?n de JWT ya emitidos mantiene su comportamiento previo.


## Correcci?n de error 500: nombre del estado de cuenta

El traceback de PostgreSQL confirm? que la columna existente es usuarios.is_active, no is_activate. Se corrigi? el mapeo ORM y el SQL local para usar is_active. El contrato del frontend no cambia. Ocho pruebas backend pasan, incluida una regresi?n que comprueba el nombre f?sico de la columna. Esta correcci?n sustituye las referencias anteriores a is_activate; no requiere renombrar la columna en Supabase.
